# 01. Data Understanding & Exploratory Data Analysis
**Project**: Next-Best-Offer and Product Recommendation with Supply Constraints

Loads the actual project datasets from `data/synthetic/`, inspects schema, missingness, distributions, and visualizes key business distributions using Matplotlib & Seaborn.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
DATA_DIR = Path('../data/synthetic')

customers     = pd.read_csv(DATA_DIR / 'customers.csv')
products      = pd.read_csv(DATA_DIR / 'products.csv')
operational   = pd.read_csv(DATA_DIR / 'operational_catalogue.csv')
transactions  = pd.read_csv(DATA_DIR / 'transactions.csv')
interactions  = pd.read_csv(DATA_DIR / 'customer_product_interactions.csv')
baskets       = pd.read_csv(DATA_DIR / 'baskets.csv')

print(f'Customers shape:          {customers.shape}')
print(f'Products shape:           {products.shape}')
print(f'Operational Cat. shape:   {operational.shape}')
print(f'Transactions shape:       {transactions.shape}')
print(f'Interactions shape:       {interactions.shape}')
print(f'Baskets shape:            {baskets.shape}')


### Column Inspection — Customers

In [ ]:
print('--- customers dtypes ---')
print(customers.dtypes)
print('\nSample:')
customers.head(3)


### Column Inspection — Products

In [ ]:
print('--- products dtypes ---')
print(products.dtypes)
products.head(3)


### Missing Values Audit

In [ ]:
for name, df in [
    ('customers', customers),
    ('products', products),
    ('operational', operational),
    ('transactions', transactions),
    ('interactions', interactions),
]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f'[{name}] — No missing values.')
    else:
        print(f'[{name}] Missing values:\n{missing}\n')


### Duplicate Row Audit

In [ ]:
for name, df in [
    ('customers', customers),
    ('products', products),
    ('operational', operational),
    ('transactions', transactions),
]:
    dups = df.duplicated().sum()
    print(f'[{name}] Duplicate rows: {dups}')


### Summary Statistics — Products and Operational Catalogue

In [ ]:
merged_prod = products.merge(operational, on='product_id')
merged_prod[['unit_price_x', 'gross_margin_pct', 'available_stock', 'shipping_days']].describe()


### Customer Segment Distribution

In [ ]:
print('Customer Segment Counts:')
print(customers['customer_segment'].value_counts())
print('\nMembership Tier Counts:')
print(customers['membership_tier'].value_counts())


### Visualizations: Customer Demographics & Membership

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Customer Membership Tier
tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']
existing_tiers = [t for t in tier_order if t in customers['membership_tier'].values]
sns.countplot(data=customers, x='membership_tier', order=existing_tiers, palette='Blues_d', ax=axes[0])
axes[0].set_title('Customer Distribution by Membership Tier', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Membership Tier')
axes[0].set_ylabel('Number of Customers')

# 2. Preferred Product Category
cat_counts = customers['preferred_category'].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette='viridis', ax=axes[1])
axes[1].set_title('Customer Preferred Category Breakdown', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Customers')
axes[1].set_ylabel('Category')

plt.tight_layout()
plt.show()


### Visualizations: Product Pricing & Supply Constraint Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Available Stock Distribution across SKUs
sns.histplot(operational['available_stock'].dropna(), bins=20, kde=True, color='#2b5c8f', ax=axes[0])
axes[0].axvline(0, color='red', linestyle='--', label='Zero Stock Threshold')
axes[0].set_title('Operational Catalogue: Available Stock Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Available Stock Units')
axes[0].set_ylabel('SKU Count')
axes[0].legend()

# 2. Gross Margin Distribution with 10% Policy Rule Cutoff
sns.histplot(operational['gross_margin_pct'].dropna() * 100, bins=15, kde=True, color='#2e7d32', ax=axes[1])
axes[1].axvline(10.0, color='red', linestyle='--', linewidth=2, label='Min Margin Policy Rule (10%)')
axes[1].set_title('Gross Margin % Distribution with Hard Policy Bound', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Gross Margin (%)')
axes[1].set_ylabel('SKU Count')
axes[1].legend()

plt.tight_layout()
plt.show()


### Data Quality Observations
- No missing values detected across customers, products, transactions, or interactions.
- Operational catalogue cleanly defines `available_stock` and `gross_margin_pct` for supply policy verification.
- Known infeasible products: `PROD_0002` (zero stock), `PROD_0036` (margin < 10%), `PROD_0044` (discontinued).
- Transactions and baskets are chronologically ordered with no future-dated entries.